# Exploración de precios de diamantes con regresión lineal

Este notebook ejecuta y revisa una canalización reproducible de regresión lineal múltiple. La variable dependiente es `price`; el modelo completo usa seis predictores numéricos y tres categóricos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import (
    DATA_PATH, TABLES_DIR, FIGURES_DIR, FULL_FORMULA,
    clean_data, data_dictionary, fit_full_model, load_data, run_pipeline
)
print(ROOT)
print(FULL_FORMULA)

## Datos, variables y limpieza

Se conserva el dataset fuente y se excluyen solo filas con dimensiones `x`, `y` o `z` no positivas.

In [ ]:
raw_data = load_data(DATA_PATH)
data, cleaning_audit = clean_data(raw_data)
print(cleaning_audit)
display(data_dictionary(data))
display(data.head())

## Modelo completo OLS

La salida incluye coeficientes, errores estándar, pruebas t, intervalos al 95 %, R², F, log-verosimilitud y diagnósticos de residuos.

In [ ]:
model = fit_full_model(data)
display(pd.DataFrame({
    'termino': model.term_names,
    'coeficiente': model.params,
    'std_err': model.bse,
    't': model.tvalues,
    'p_value': model.pvalues,
}))

## Comparación de pares, significancia y homocedasticidad

Se ajustan los 36 modelos posibles con dos variables predictoras originales. Las categóricas se codifican mediante indicadores; por ello, los modelos pueden tener diferente número de parámetros. También se identifican los términos significativos y se aplica Breusch-Pagan al modelo completo.

In [ ]:
run_summary = run_pipeline()
run_summary

In [ ]:
display(pd.read_csv(TABLES_DIR / 'full_model_metrics.csv'))
display(pd.read_csv(TABLES_DIR / 'full_model_coefficients.csv'))
display(pd.read_csv(TABLES_DIR / 'two_predictor_models.csv').head())
display(pd.read_csv(TABLES_DIR / 'significant_terms_5pct.csv'))
display(pd.read_csv(TABLES_DIR / 'categorical_variables_joint_significance.csv'))
display(pd.read_csv(TABLES_DIR / 'top_two_numeric_predictors_by_abs_t.csv'))
display(pd.read_csv(TABLES_DIR / 'top_two_coefficients_by_abs_t.csv'))
display(pd.read_csv(TABLES_DIR / 'selected_predictor_ci_95.csv'))
display(pd.read_csv(TABLES_DIR / 'breusch_pagan.csv'))

## Figuras de diagnóstico

Las gráficas ayudan a explorar la relación principal y los supuestos del modelo lineal.

In [ ]:
for name in ['price_vs_carat.png', 'residuals_vs_fitted.png', 'residuals_qq.png']:
    display(Image(filename=FIGURES_DIR / name))